# Assignment Instructions - Part 4: Reproducible AutoML Arrays on the HPCC

> **Prerequisite:** Complete Part 3's local TPOT workflow before beginning this HPCC activity. Confirm your HPCC account, partition, and Conda instructions with the instructor.

Part 3 ran one bounded TPOT search. Part 4 asks a more reliable question: *how sensitive is an evolutionary AutoML search to its random seed?* You will run the same CASH problem several times in a Slurm job array, preserve a record from each run, and compare the recommendations.

The goal is **not** to select the run with the largest test score. That would reuse the final test set during model selection. Instead, the array summary selects a candidate using cross-validation results from the training data; one final command evaluates that already-chosen candidate on the sealed test set.

## Learning goals

By the end, you should be able to:

- create and document a portable TPOT environment on the HPCC;
- turn one reproducible AutoML experiment into a bounded Slurm array;
- collect machine-readable per-run records; and
- distinguish selecting among candidates from final held-out evaluation; and
- reflect on the evidence and infrastructure a teammate would need to audit the experiment.

## 1. Workflow and resource contract

Each array task uses the **same data split**, metric, TPOT search space, cross-validation setup, and runtime budget. Only `search_seed` changes. Keeping the split fixed makes seed-to-seed differences easier to attribute to TPOT's evolutionary search rather than to different training data.

| Phase | Input | Output | Test data used? |
|---|---|---|---|
| Smoke test | one seed, training data | one saved pipeline and JSON record | No |
| Array search | several search seeds, training data | one directory per seed | No |
| Summary | saved JSON records | ranked candidate and selection record | No |
| Final evaluation | selected candidate only | one held-out score and confusion matrix | Yes, once |

Start small: 4 seeds, 5 minutes per seed, 1 CPU per task. Increasing the array size or time budget expands the AutoML search and the computing cost; record that change as part of the experiment.

## 2. Install a reproducible environment on the HPCC

On this HPCC, first make the Miniforge Conda tools available:

```bash
module purge
module load Miniforge3
```

From a clone of this repository, create the environment from the pinned course specification:

```bash
conda env create --prefix ./envs --file environment.yml
conda activate ./envs
python -c "import sklearn, tpot; print(sklearn.__version__, tpot.__version__)"
```

If you must install interactively, record the exact command and package versions in your experiment notes. Do **not** repeatedly install packages inside every Slurm task.

TPOT itself can schedule work. This framework requests one Slurm CPU and sets TPOT's `n_jobs=1` so each array task uses only the resource it requested. Ask the HPCC team before changing either setting.

## 3. Create the array worker

Run the next cell once in a writable copy of the repository. It creates `run_tpot_array.py`. The worker:

- rebuilds the Part 3 breast-cancer classification problem;
- makes a fixed training/test split but searches only on the training data;
- runs TPOT with one search seed;
- saves the fitted pipeline and a JSON record; and
- intentionally does **not** calculate a test score.

Read the script before launching it. For your own data, replace the data-loading block but keep target definition, split logic, preprocessing, and provenance explicit.

In [1]:
%%writefile run_tpot_array.py
import argparse
import json
from pathlib import Path

import joblib
import sklearn
import tpot
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from tpot import TPOTClassifier


def parse_args():
    parser = argparse.ArgumentParser(
        description="Run one bounded TPOT search and save its training-only record."
    )
    parser.add_argument("--search-seed", type=int, required=True)
    parser.add_argument("--split-seed", type=int, default=802)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-time-mins", type=float, default=5)
    return parser.parse_args()


def main():
    args = parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=False)

    dataset = load_breast_cancer(as_frame=True)
    X_train, _, y_train, _ = train_test_split(
        dataset.data,
        dataset.target,
        test_size=0.20,
        stratify=dataset.target,
        random_state=args.split_seed,
    )

    search = TPOTClassifier(
        search_space="linear",
        scorers=["balanced_accuracy"],
        cv=5,
        max_time_mins=args.max_time_mins,
        max_eval_time_mins=min(1.0, args.max_time_mins / 2),
        n_jobs=1,
        random_state=args.search_seed,
        verbose=5,
    )
    search.fit(X_train, y_train)

    records = search.evaluated_individuals
    best_cv_score = float(records["balanced_accuracy_score"].max())
    pipeline_path = args.output_dir / "pipeline.joblib"
    joblib.dump(search.fitted_pipeline_, pipeline_path)

    record = {
        "search_seed": args.search_seed,
        "split_seed": args.split_seed,
        "max_time_mins": args.max_time_mins,
        "search_space": "linear",
        "metric": "balanced_accuracy",
        "cv_folds": 5,
        "n_jobs": 1,
        "n_training_observations": len(X_train),
        "n_features": X_train.shape[1],
        "candidate_count": len(records),
        "best_cv_score": best_cv_score,
        "pipeline_path": str(pipeline_path),
        "pipeline_repr": repr(search.fitted_pipeline_),
        "scikit_learn_version": sklearn.__version__,
        "tpot_version": tpot.__version__,
    }
    (args.output_dir / "run.json").write_text(json.dumps(record, indent=2) + "\n")
    records.to_csv(args.output_dir / "evaluated_individuals.csv", index=False)
    print(json.dumps(record, indent=2))


if __name__ == "__main__":
    main()

Writing run_tpot_array.py


## 4. Smoke test before the array

Run the exact worker command below from an interactive HPCC session or a short single Slurm job. Inspect `results/smoke-test/run.json` and confirm that it contains the search seed, fixed split seed, candidate count, package versions, a cross-validation score, and **no held-out test score**.

Delete the smoke-test directory before re-running the same command: the script refuses to overwrite an existing run directory to prevent silent loss of evidence.

In [2]:
# Run in an HPCC terminal, not as a notebook Python cell:
#
# conda activate ./envs
 python run_tpot_array.py \
   --search-seed 802 \
   --split-seed 802 \
   --max-time-mins 2 \
   --output-dir results/smoke-test

## 5. Create and submit the Slurm array

Change only the marked site-specific values after consulting your HPCC documentation. The sample uses four tasks, with seeds 802–805. It requests one CPU per task and uses an output folder that includes the seed, so tasks cannot overwrite each other.

The `--time` allocation must exceed TPOT's internal search budget because environment startup, data loading, and result writing also take time.

In [3]:
%%writefile submit_tpot_array.sb
#!/bin/bash --login
#SBATCH --array=0-3
#SBATCH --cpus-per-task=1
#SBATCH --mem=8G
#SBATCH --time=01:00:00

set -euo pipefail # Force quit if there is an error.

module purge
module load Miniforge3
conda activate ./envs

SEARCH_SEED=$((802 + SLURM_ARRAY_TASK_ID))
OUTPUT_DIR="results/seed-${SEARCH_SEED}"

python run_tpot_array.py \
  --search-seed "${SEARCH_SEED}" \
  --split-seed 802 \
  --max-time-mins 5 \
  --output-dir "${OUTPUT_DIR}"


Writing submit_tpot_array.sb


In [ ]:
# Run in an HPCC terminal:
#
# sbatch submit_tpot_array.sb
#
# Monitor with commands approved for your cluster, for example:
# squeue --me
# sacct -j JOB_ID --format=JobID,State,Elapsed,AllocCPUS,MaxRSS

### Before moving on

- Did every task finish successfully? Inspect the `.out` and `.err` logs, not just the scheduler status.
- Did each task create exactly one `results/seed-*/run.json` file?
- Did you accidentally use more than one CPU per task or call `pip install` from the batch script?
- If one seed fails, fix the root cause and rerun that seed in a new, explicitly named output directory. Do not hide the failure by editing or deleting evidence.

## 6. Summarize the completed searches

The next cell writes a comparison script. It reads the saved JSON records, ranks runs by the **training-only cross-validation** score, and writes `results/selection.json`. The word “best” here has a deliberately narrow meaning: highest reported CV score among the bounded searches run with this configuration. It does not mean clinically best, scientifically true, most interpretable, or guaranteed to generalize.

Before accepting the candidate, inspect whether score differences are meaningful, whether pipeline structures are stable across seeds, and whether a simpler baseline remains preferable.

In [ ]:
%%writefile summarize_tpot_runs.py
import argparse
import json
from pathlib import Path

import pandas as pd


def parse_args():
    parser = argparse.ArgumentParser(
        description="Rank completed TPOT array runs using training-only CV scores."
    )
    parser.add_argument("--results-dir", type=Path, default=Path("results"))
    return parser.parse_args()


def main():
    args = parse_args()
    record_paths = sorted(args.results_dir.glob("seed-*/run.json"))
    if not record_paths:
        raise FileNotFoundError(
            f"No completed run records found in {args.results_dir}/seed-*/run.json"
        )

    records = [json.loads(path.read_text()) for path in record_paths]
    table = pd.DataFrame(records).sort_values(
        "best_cv_score",
        ascending=False,
        ignore_index=True,
    )
    required = {"search_seed", "split_seed", "best_cv_score", "pipeline_path"}
    missing = required.difference(table.columns)
    if missing:
        raise ValueError(f"Run records are missing required fields: {sorted(missing)}")
    if table["split_seed"].nunique() != 1:
        raise ValueError("Cannot compare runs that used different train/test splits.")

    table.to_csv(args.results_dir / "array_summary.csv", index=False)
    best = table.iloc[0].to_dict()
    selection = {
        "selection_rule": (
            "Highest training-only 5-fold cross-validation balanced accuracy "
            "among completed runs with the same fixed split."
        ),
        "selected_run": best,
        "warning": (
            "This record was selected without using the held-out test set. "
            "Evaluate it once, after reviewing stability and interpretability."
        ),
    }
    (args.results_dir / "selection.json").write_text(
        json.dumps(selection, indent=2) + "\n"
    )

    print(table[["search_seed", "candidate_count", "best_cv_score"]].to_string(index=False))
    print(f"\nCandidate selected for one final test evaluation: seed {best['search_seed']}")


if __name__ == "__main__":
    main()

In [ ]:
# Run in an HPCC terminal after all array tasks succeed:
#
# python summarize_tpot_runs.py --results-dir results
#
# Then inspect:
# - results/array_summary.csv
# - results/selection.json
# - each pipeline_repr in the summary
# - each task's logs and evaluated_individuals.csv

## 7. Evaluate the selected candidate once

Only after documenting why the selected candidate is acceptable should you open the final test set. This script re-creates the fixed split, loads the candidate chosen in `selection.json`, and reports its balanced accuracy and a confusion matrix. Do not run it once per seed and retain the largest score.

In [ ]:
%%writefile evaluate_selected_tpot.py
import argparse
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.model_selection import train_test_split


def parse_args():
    parser = argparse.ArgumentParser(
        description="Evaluate one previously selected TPOT pipeline on the sealed test set."
    )
    parser.add_argument("--selection", type=Path, default=Path("results/selection.json"))
    return parser.parse_args()


def main():
    args = parse_args()
    selection = json.loads(args.selection.read_text())
    candidate = selection["selected_run"]
    pipeline = joblib.load(candidate["pipeline_path"])

    dataset = load_breast_cancer(as_frame=True)
    _, X_test, _, y_test = train_test_split(
        dataset.data,
        dataset.target,
        test_size=0.20,
        stratify=dataset.target,
        random_state=candidate["split_seed"],
    )
    predictions = pipeline.predict(X_test)
    score = balanced_accuracy_score(y_test, predictions)
    print(f"Held-out balanced accuracy: {score:.3f}")
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        display_labels=dataset.target_names,
        cmap="Blues",
    )
    plt.title(f"Selected TPOT pipeline (seed {candidate['search_seed']})")
    plt.show()


if __name__ == "__main__":
    main()

## 8. Required evidence and final reflection

Include the following in your project repository or project notes: the Slurm script, a `run.json` record for each successful seed, the array summary, the selection record, and the command or job identifier used for the one final held-out evaluation. Do not include sensitive account information, credentials, or private data.

Write a short reflection in your project repository as part of the final paper or project notes. Answer each prompt:

1. What varied across your array tasks, and what remained fixed? Why is that distinction important?
2. Was the selected pipeline clearly better than the other runs, or could its lead plausibly be search noise?
3. How would the conclusion change if you ranked runs by their final test score instead of their training-only CV score?
4. What resource settings, software versions, input data details, and output artifacts would a future teammate need to reproduce this experiment?
5. Which of Safe, Portable, Reproducible, Robust, and Literate was hardest to achieve in this workflow? Identify one concrete improvement you would make before reusing it for research.

This reflection is part of the required assignment and should be submitted with the rest of your project work.

## Design intent

**Primary goal:** connect AutoML search to a reproducible HPCC workflow.  
**Support goals:** practice bounded parallel experimentation, evidence-preserving handoff, and responsible selection.  
**Prerequisites:** Part 3's CASH, pipelines, baseline, and held-out-test concepts; basic Slurm access.  
**Prepares students for:** larger research workflows in which many model candidates must be compared without silently overfitting the evaluation data.  
**If omitted:** students still learn TPOT, but miss how search variability and batch infrastructure affect trust in an AutoML result.